In [5]:
# Run and print a shell command.
def run(cmd):
  print('>> {}'.format(cmd))
  !{cmd}
  print('')

# Install apache-beam.
run('pip install --quiet "pandas<2.1.0" "numpy<2.0.0" apache-beam --force-reinstall')

>> pip install --quiet "pandas<2.1.0" "numpy<2.0.0" apache-beam --force-reinstall



ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Hritvik\\Desktop\\Northeastern University\\IE7374 MLOps\\Labs\\MLOps\\venv\\Lib\\site-packages\\~standard\\backend_c.cp311-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import apache_beam as beam
import re
import os

# 1. Path to pick up all .txt files in the data folder
data_dir = r"C:\Users\Hritvik\Desktop\Northeastern University\IE7374 MLOps\Labs\MLOps\Labs\Data_Labs\Apache_Beam_Labs\data"
inputs_pattern = os.path.join(data_dir, "*.txt")
outputs_prefix = 'outputs/combined_count'

# # Setup: Download two different files
def setup_data():
    !mkdir -p data
    # File 1: Alice in Wonderland
    !curl -L https://www.gutenberg.org/cache/epub/11/pg11.txt -o data/alice.txt
    # File 2: Through the Looking Glass
    !curl -L https://www.gutenberg.org/cache/epub/12/pg12.txt -o data/glass.txt

setup_data()


In [2]:
print(inputs_pattern)

C:\Users\Hritvik\Desktop\Northeastern University\IE7374 MLOps\Labs\MLOps\Labs\Data_Labs\Apache_Beam_Labs\data\*.txt


In [ ]:

with beam.Pipeline() as pipeline:
    word_counts = (
        pipeline
        | 'Read multiple files' >> beam.io.ReadFromText(inputs_pattern)
        
        # Transformation Logic: Find words AND convert to lowercase
        | 'Extract & Lowercase' >> beam.FlatMap(lambda line: re.findall(r"[a-zA-Z']+", line.lower()))
        
        | 'Pair with 1' >> beam.Map(lambda word: (word, 1))
        | 'Group and sum' >> beam.CombinePerKey(sum)
    )

    (
        word_counts
        | 'Format results' >> beam.Map(lambda wc: f"{wc[0]}: {wc[1]}")
        | 'Write results' >> beam.io.WriteToText(outputs_prefix)
    )

# Inspect the results from the combined files
!powershell -command "Get-Content outputs/combined_count-00000-of-* -TotalCount 20"

the: 3639
project: 176
gutenberg: 196
ebook: 26
of: 1256
alice's: 4
adventures: 12
in: 907
wonderland: 8
this: 349
is: 280
for: 405
use: 61
anyone: 10
anywhere: 5
united: 30
states: 38
and: 1928
most: 32
other: 141
